In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 5 Extreme Logistic Regressor with Majority Undersampling & Class Weighting (`models/lr_feng_esi5_extreme.ipynb`)

This notebook trains a **Binary Logistic Regressor** for **ESI 5 vs Not ESI 5 (with ESI 1 excluded)** with **Majority Class Undersampling**, **Inverse Class Frequency Weighting**, and **Iterative Weight Multiplier Optimization**:

### System Architecture & Key Features
1. **ESI 1 Excluded & Majority Undersampling**: ESI 1 rows are excluded, and majority class (`"not_5"`) rows are undersampled (e.g. 1:1 ratio or configurable ratio/percentage) to balance class distributions.
2. **Feature Set (13 Clinical Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and 10 vital sign anomaly flags.
3. **Inverse Class Frequency Weighting**: Calculates baseline inverse class weight $w_{\text{inv}} = N_{\text{non-ESI 1}} / N_{\text{ESI 5}}$.
4. **Iterative Weight Multiplier Benchmark Grid**: Tunes class weight multipliers over a grid to evaluate Precision and Recall trade-offs.
5. **Reports & Artifacts**:
   - **Plots**: `plots/lr_feng_esi5_weight_tuning.png` (Precision & Recall per iteration).
   - **CSV Reports**: `reports/lr_feng_esi5_tuning_results.csv`, `reports/lr_feng_esi5_val_report.csv`, `reports/lr_feng_esi5_test_report.csv`.
   - **Model Export**: Saved to `deploy/lr_feng_esi5_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Exclude ESI 1 Rows, Compute 13 FE Inputs & Apply Majority Class Undersampling
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Compute 13 Clinical Feature Engineering flags
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

df_feng[[target_col]] <- raw_df[[target_col]]

# Filter out ESI 1 rows completely
raw_esi <- as.character(df_feng[[target_col]])
non_esi1_idx <- which(raw_esi != "1" & !is.na(raw_esi))
df_feng <- df_feng[non_esi1_idx, ]
raw_esi_sub <- raw_esi[non_esi1_idx]

# Create Binary ESI 5 Target: '5' vs 'not_5'
df_feng$target_layer2 <- factor(ifelse(raw_esi_sub == "5", "5", "not_5"), levels = c("5", "not_5"))

initial_rows <- nrow(df_feng)
df_feng <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining: %d)\n", initial_rows - nrow(df_feng), nrow(df_feng)))
# ---------------------------------------------------------
# MAJORITY CLASS UNDERSAMPLING ('not_5')
# Set undersample_ratio: 1.0 for 1:1 balanced undersampling, or custom ratio/percentage
# ---------------------------------------------------------
undersample_mode  <- "ratio"  # "ratio" (1:N majority vs minority) or "percentage"
undersample_ratio <- 1.0      # 1:1 balanced majority undersampling

idx_5     <- which(df_feng$target_layer2 == "5")
idx_not_5 <- which(df_feng$target_layer2 == "not_5")

if (undersample_mode == "ratio") {
  n_not_5_keep <- min(length(idx_not_5), round(length(idx_5) * undersample_ratio))
  kept_not_5   <- sample(idx_not_5, size = n_not_5_keep)
  kept_5       <- idx_5
} else {
  keep_ratio_not_5 <- 1.00
  kept_not_5       <- sample(idx_not_5, size = round(length(idx_not_5) * keep_ratio_not_5))
  kept_5           <- idx_5
}

df_feng <- df_feng[sort(c(kept_5, kept_not_5)), ]

cat(sprintf("Undersampled Binary ESI 5 Dataset Ready (Minority '5': %d, Undersampled Majority 'not_5': %d): %d total rows x %d cols\n",
            length(kept_5), length(kept_not_5), nrow(df_feng), ncol(df_feng)))
cat("Undersampled Binary Target Distribution:\n")
print(table(df_feng$target_layer2))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Continuous Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_layer2, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer2, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Inverse Class Frequency Weighting & Iterative Class Weight Optimization Grid
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Calculate Base Inverse Class Frequency Weight for ESI 5
n_total <- nrow(train_df)
n_esi5  <- sum(train_df$target_layer2 == "5")
inv_weight_esi5 <- n_total / n_esi5

cat(sprintf("=== Inverse Class Frequency Weight Calculation ===\n"))
cat(sprintf("  - ESI 5 Count: %d / %d -> Base Inverse Weight: %.2f\n\n", n_esi5, n_total, inv_weight_esi5))
# Define Iterative Weight Multipliers Grid
multipliers <- c(0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0)
feat_names  <- setdiff(names(train_df), c(target_col, "target_layer2"))
formula_lr  <- as.formula(paste("target_layer2 ~", paste(feat_names, collapse = " + ")))
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
results_list <- list()
cat(sprintf("Starting Iterative Class Weight Optimization over %d grid iterations...\n", length(multipliers)))
for (iter in 1:length(multipliers)) {
  m_val <- multipliers[iter]
  w_eff <- m_val * inv_weight_esi5
  
  weights <- ifelse(train_df$target_layer2 == "5", w_eff, 1.0)
  model   <- multinom(formula_lr, data = train_df, weights = weights, trace = FALSE, MaxNWts = 5000)
  
  pred_val <- as.character(predict(model, newdata = val_df))
  pred_val[is.na(pred_val)] <- "not_5"
  pred_fac <- factor(pred_val, levels = c("5", "not_5"))
  act_fac  <- factor(val_df$target_layer2, levels = c("5", "not_5"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "5")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1 <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  prob_res <- predict(model, newdata = val_df, type = "probs")
  prob_5   <- if (is.matrix(prob_res)) {
    if ("5" %in% colnames(prob_res)) prob_res[, "5"] else 1 - prob_res[, "not_5"]
  } else {
    1 - prob_res
  }
  pr_auc <- calc_pr_auc(ifelse(act_fac == "5", 1, 0), prob_5)
  
  results_list[[iter]] <- data.frame(
    Iteration    = iter,
    Multiplier   = m_val,
    Class_Weight = round(w_eff, 2),
    Accuracy     = round(acc, 4),
    Precision    = round(prec, 4),
    Recall       = round(rec, 4),
    F1_Score     = round(f1, 4),
    PR_AUC       = round(pr_auc, 4)
  )
}
tuning_df <- do.call(rbind, results_list)
# Write Tuning Results to CSV
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
csv_tuning_path <- file.path(reports_dir, "lr_feng_esi5_tuning_results.csv")
write.csv(tuning_df, file = csv_tuning_path, row.names = FALSE)
cat("Tuning Results CSV written to:", csv_tuning_path, "\n\n")
cat("Iterative Weight Optimization complete! Top 5 configurations by Validation F1 Score:\n")
print(head(tuning_df[order(-tuning_df$F1_Score), ], 5))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Save Trajectory Diagnostic Plot (Precision & Recall per Iteration)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
df_plot <- tuning_df %>%
  select(Iteration, Precision, Recall, F1_Score, PR_AUC) %>%
  pivot_longer(cols = c("Precision", "Recall", "F1_Score", "PR_AUC"), names_to = "Metric", values_to = "Score")
p1 <- ggplot(df_plot, aes(x = Iteration, y = Score, color = Metric)) +
  geom_line(size = 1.0) + geom_point(size = 2.0) +
  theme_minimal() +
  scale_color_manual(values = c("Precision" = "#1d3557", "Recall" = "#457b9d", "F1_Score" = "#2b5c8f", "PR_AUC" = "#81b29a")) +
  labs(title = "Binary ESI 5: Precision, Recall & F1 Trajectories Across Class Weight Iterations",
       subtitle = "Evaluating performance trajectory over varying inverse weight multipliers",
       x = "Tuning Iteration", y = "Metric Value Score") +
  theme(plot.title = element_text(face = "bold", size = 12), legend.position = "top")
plot_path <- file.path(plots_dir, "lr_feng_esi5_weight_tuning.png")
ggsave(plot_path, plot = p1, width = 8.5, height = 4.5, dpi = 300)
cat("ESI 5 Weight Tuning Plot saved to:", plot_path, "\n")
p1

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Retrain Optimal Model & Export CSV Reports (Actual vs Predicted Count, Precision, Recall, PR-AUC)
# ---------------------------------------------------------
best_row <- tuning_df[which.max(tuning_df$F1_Score), ]
cat(sprintf("=== Selected Optimal Weight Configuration (Iteration %d) ===\n", best_row$Iteration))
cat(sprintf("  Multiplier       : %.2f\n", best_row$Multiplier))
cat(sprintf("  Effective Weight : %.2f\n", best_row$Class_Weight))
cat(sprintf("  Validation F1    : %.4f\n", best_row$F1_Score))
cat(sprintf("  Validation Prec  : %.4f\n", best_row$Precision))
cat(sprintf("  Validation Rec   : %.4f\n\n", best_row$Recall))
final_weights <- ifelse(train_df$target_layer2 == "5", best_row$Class_Weight, 1.0)
lr_esi5_final <- multinom(formula_lr, data = train_df, weights = final_weights, trace = FALSE, MaxNWts = 5000)
evaluate_and_report_binary_esi5 <- function(model, data, set_name) {
  pred_val <- as.character(predict(model, newdata = data))
  pred_val[is.na(pred_val)] <- "not_5"
  pred_fac <- factor(pred_val, levels = c("5", "not_5"))
  act_fac  <- factor(data$target_layer2, levels = c("5", "not_5"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "5")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  prob_res <- predict(model, newdata = data, type = "probs")
  prob_5   <- if (is.matrix(prob_res)) {
    if ("5" %in% colnames(prob_res)) prob_res[, "5"] else 1 - prob_res[, "not_5"]
  } else {
    1 - prob_res
  }
  pr_auc <- calc_pr_auc(ifelse(act_fac == "5", 1, 0), prob_5)
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = c("5", "not_5"),
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(c(prec, ifelse(is.na(cm$byClass["Neg Pred Value"]), 0, cm$byClass["Neg Pred Value"])), 4),
    Recall       = round(c(rec, ifelse(is.na(cm$byClass["Specificity"]), 0, cm$byClass["Specificity"])), 4),
    PR_AUC       = round(c(pr_auc, NA), 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   BINARY ESI 5 OPTIMAL LOGISTIC REGRESSOR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ESI 5 Precision      : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  ESI 5 Recall (Sens)  : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  ESI 5 F1 Score       : %.4f\n", f1))
  cat(sprintf("  ESI 5 PR-AUC         : %.4f\n", pr_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Class Count Comparison & Metrics Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(report_df)
}
# Generate Validation Report & Write to CSV
val_report <- evaluate_and_report_binary_esi5(lr_esi5_final, val_df, "Validation")
csv_val_path <- file.path(reports_dir, "lr_feng_esi5_val_report.csv")
write.csv(val_report, file = csv_val_path, row.names = FALSE)
cat("Validation CSV Report written to:", csv_val_path, "\n")
# Generate Test Report & Write to CSV
test_report <- evaluate_and_report_binary_esi5(lr_esi5_final, test_df, "Test")
csv_test_path <- file.path(reports_dir, "lr_feng_esi5_test_report.csv")
write.csv(test_report, file = csv_test_path, row.names = FALSE)
cat("Test CSV Report written to:", csv_test_path, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Optimal Binary ESI 5 Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "lr_feng_esi5_extreme_model.rds")
saveRDS(list(model = lr_esi5_final, preproc = preproc), file = model_path)
cat("Optimal Binary ESI 5 Logistic Regressor model saved to:", model_path, "\n")